# Census graphs: shared boundaries and Euclidean neighbours

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from padova_core.census_graphs import (
    read_order, read_zones, boundary_pairs, undirected_pairs,
    euclidean_knn, restrict_pairs, bidirectional,
)

PROJECT_ROOT = Path.home() / "padova_data"
BASE = PROJECT_ROOT / "gnn_outputs"
GEOMETRY = PROJECT_ROOT / "Densità della popolazione/SIT_SEZIONI_2021/Sez_censimento_2021_32632_fixed.gpkg"
NODE_INDEX = BASE / "node_index.csv"
HISTORICAL = BASE / "edge_index_contiguity.npy"

SAMPLE_ORDER = None
LAYER = None
MIN_SHARED_M = 0.0
K = 8

if SAMPLE_ORDER is None:
    OUT = BASE / "census_graph_comparison" / "full"
else:
    OUT = BASE / "census_graph_comparison" / Path(SAMPLE_ORDER).parent.name

OUT.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT)

## Read census geometry and align section IDs

In [ ]:
ids = read_order(NODE_INDEX, "node_id")
zones = read_zones(GEOMETRY, ids, LAYER)
rebuilt = boundary_pairs(zones, MIN_SHARED_M)

has_reference = HISTORICAL is not None and Path(HISTORICAL).exists()
if has_reference:
    raw_edges = np.load(HISTORICAL, allow_pickle=False)
    full_pairs = undirected_pairs(raw_edges, len(ids))
    source = "historical"
else:
    full_pairs = rebuilt
    source = "geometry_reconstruction"

print(f"Nodes: {len(ids)}, source: {source}, edges: {len(full_pairs)}")

# Check differences between historical edges and rebuilt edges
diff_rows = []
if has_reference:
    old_set = set([tuple(p) for p in full_pairs])
    new_set = set([tuple(p) for p in rebuilt])

    for i, j in sorted(old_set - new_set):
        diff_rows.append(["historical_only", ids[i], ids[j]])
    for i, j in sorted(new_set - old_set):
        diff_rows.append(["rebuilt_only", ids[i], ids[j]])

diff_df = pd.DataFrame(diff_rows, columns=["difference", "SEZ21_ID_u", "SEZ21_ID_v"])
diff_df.to_csv(OUT / "boundary_differences.csv", index=False, encoding="utf-8-sig")
print("Number of different edges:", len(diff_df))

## Construct both graphs on the same nodes

In [ ]:
if SAMPLE_ORDER is None:
    selected = ids
else:
    selected = read_order(SAMPLE_ORDER, "local_node_id")

# Get centroid coordinates
all_xy = []
for g in zones:
    p = g.Centroid()
    all_xy.append([p.GetX(), p.GetY()])
all_xy = np.array(all_xy)

# Match coordinates to the selected node order
id_to_idx = {key: i for i, key in enumerate(ids)}
sel_idx = [id_to_idx[key] for key in selected]
xy = all_xy[sel_idx]

# Build shared boundary graph and Euclidean kNN graph
confine = restrict_pairs(full_pairs, ids, selected)
euclidean, distance_m, weights, sigma = euclidean_knn(xy, K)

# Save node orders and centroids
pd.DataFrame({
    "local_node_id": range(len(selected)),
    "SEZ21_ID": selected
}).to_csv(OUT / "node_order.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "node_id": range(len(ids)),
    "SEZ21_ID": ids
}).to_csv(OUT / "full_node_order.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "node_id": range(len(ids)),
    "SEZ21_ID": ids,
    "x": all_xy[:, 0],
    "y": all_xy[:, 1],
    "crs_epsg": 32632
}).to_csv(OUT / "zone_centroids.csv", index=False, encoding="utf-8-sig")

# Save npy files for GNN
np.save(OUT / "edge_index_contiguity.npy", bidirectional(full_pairs))
np.save(OUT / "edge_index_contiguity_static.npy", bidirectional(confine))
np.save(OUT / f"edge_index_euclidean_k{K}.npy", bidirectional(euclidean))
np.save(OUT / f"edge_weight_euclidean_k{K}.npy", np.concatenate([weights, weights]))

# Save Euclidean edge list
edge_rows = []
for (i, j), d, w in zip(euclidean, distance_m, weights):
    edge_rows.append([i, j, selected[i], selected[j], d, w])

edges_df = pd.DataFrame(
    edge_rows,
    columns=["local_u", "local_v", "SEZ21_ID_u", "SEZ21_ID_v", "distance_m", "weight"]
)
edges_df.to_csv(OUT / "euclidean_edges.csv", index=False, encoding="utf-8-sig")

# Save graph summary
summary = {
    "boundary_source": source,
    "min_shared_m": MIN_SHARED_M,
    "full_nodes": len(ids),
    "reference_edge_differences": len(diff_df) if has_reference else None,
    "sample_nodes": len(selected),
    "confine_edges": len(confine),
    "euclidean_edges": len(euclidean),
    "k": K,
    "sigma_m": sigma,
    "sample_order": str(SAMPLE_ORDER) if SAMPLE_ORDER is not None else str(NODE_INDEX),
    "crs_epsg": 32632,
}

with open(OUT / "graph_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(summary)

## Plot the two graphs at a common scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharex=True, sharey=True)

# Left: shared boundary graph
axes[0].add_collection(LineCollection(xy[confine], colors="0.62", linewidths=0.45))
axes[0].scatter(xy[:, 0], xy[:, 1], s=2, color="blue")
axes[0].set_aspect("equal")
axes[0].set_axis_off()
axes[0].set_title(f"Confine condiviso · {len(selected)} nodi, {len(confine)} archi")

# Right: Euclidean kNN graph
axes[1].add_collection(LineCollection(xy[euclidean], colors="0.62", linewidths=0.45))
axes[1].scatter(xy[:, 0], xy[:, 1], s=2, color="blue")
axes[1].set_aspect("equal")
axes[1].set_axis_off()
axes[1].set_title(f"Distanza euclidea · {len(selected)} nodi, {len(euclidean)} archi")

plt.tight_layout()
plt.savefig(OUT / "confine_euclidea.png", dpi=220, bbox_inches="tight")
plt.show()